<a href="https://colab.research.google.com/github/SyameimaruKoa/Minecraft-mod-AutoTranslation-tool/blob/main/run_colab_server.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Minecraft Mod Auto Translator - Google Colab (LM Link)

このノートブックは、Google Colab 上で LM Studio (llmster) をバックグラウンドで起動し、**LM Link** を介してローカルPCの LM Studio と接続するためのものじゃ。

### 💡 LM Link の仕組みとメリット
- **ローカルPCで一元管理**: モデルのダウンロード、ロード、GPUオフロード設定などはすべて、そなたのローカルPCの LM Studio GUI からコントロールできるぞ。
- **設定不要**: Tailscale の認証キー登録や、面倒なポート開放設定などは一切不要じゃ。
- **シームレスな連携**: Colab とローカルPCをリンクすると、ローカルPCのモデル一覧にリモートモデルが表示される。ローカルの翻訳ツールは `localhost:1234`（デフォルト設定）に向けてリクエストを投げるだけで、自動的に Colab の GPU で推論が行われるのじゃ。

## 1. LM Studio (Headless版) のインストール

まずは Google Colab 上に LM Studio の CLI / デーモンをインストールするのじゃ。

In [ ]:
import os, subprocess, time
# 既存の古いデーモンプロセスを完全に終了（パスキー不一致による認証エラーを防ぐため）
!pkill -f lms || true
!pkill -f llmster || true
!pkill -f lmstudio || true
# Headless版 LM Studio のインストール
!curl -fsSL https://lmstudio.ai/install.sh | bash
# 環境変数PATHにlmsの実行フォルダを追加
os.environ["PATH"] = os.path.expanduser("~/.lmstudio/bin:") + os.environ["PATH"]
# パスなどのブートストラップを実行
!lms bootstrap
# ディスク書き込みを同期(Text file busyエラー防止)
!sync
# デーモン(llmster)をバックグラウンド起動
subprocess.Popen(["lms", "daemon", "up"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
print("[INFO] LM Studio Daemon (llmster) 起動完了じゃ。")

## 2. LM Link のログインと接続有効化

下のセルを実行すると、ログイン認証用のURLが表示されるぞ。
1. 表示されたURLをクリックし、ブラウザで LM Studio アカウントにログインして連携を承認するのじゃ。
2. 承認後、Colab上のLM Studioと、そなたのローカルPC上のLM Studioが自動的にリンクされるぞ。
3. リンク完了後は、ブラウザやこのColabタブは開いたままにしておくのじゃぞ（Colabインスタンスが起動し続けるため）。

In [ ]:
# アカウントログイン
!lms login
# LM Linkの有効化
!lms link enable

## 3. PLaMo 3 Translateのモデル選択・ダウンロード・ロード

4サイズ（0.8B / 2B / 8B / 31B）から1つ選び、GGUF量子化形式を指定するのじゃ。
Colabでは**選んだモデル1つだけ**をロードする。選び直してこのセルを再実行すると、ロード済みのモデルをアンロードして切り替えるぞ。
ダウンロード済みモデルは再利用できるが、Colabランタイムを初期化すると保存データが消えることがあるのじゃ。

31Bなど大きなモデルはColabの割り当てGPU・メモリによってはロードできぬ。

In [ ]:
import subprocess

plamo_size = "2b" # @param ["0.8b", "2b", "8b", "31b"]
quantization = "Q4_K_M" # @param ["Q4_K_M", "Q3_K_M", "Q2_K", "Q4_0"]

if quantization == "Q4_0" and plamo_size not in ("0.8b", "2b"):
    raise ValueError("Q4_0は0.8Bと2Bのみ選択できるのじゃ")

repo = f"pfnet/plamo-3-translate-{plamo_size}-GGUF"
model_key = f"{repo}@{quantization.lower()}"
model_id = f"plamo-3-translate-{plamo_size}"
print(f"ダウンロード対象: {model_key}", flush=True)
subprocess.run(["lms", "get", model_key, "--gguf"], check=True, stdin=subprocess.DEVNULL)
print("既存モデルをアンロードして1モデルだけロードするのじゃ", flush=True)
subprocess.run(["lms", "unload", "--all"], check=True)
subprocess.run(["lms", "load", model_key, "--gpu=max", f"--identifier={model_id}"], check=True)
subprocess.run(["lms", "server", "start", "--port", "1234"], check=True)
subprocess.run(["lms", "ps"], check=True)
print(f"ロード完了: {model_id}（Windows側のPLaMoモードから選択可能）", flush=True)
